In [1]:
import networkx as nx
import numpy as np
from collections import Counter
import pandas as pd
import os
from graph import Graph
from utils import setup_logger
from nested_cv_fgw_multi import nested_fgw_multiclass
from nested_cv_grakel_multi import nested_cv_multiclass

In [2]:
def convert_nx_to_grakel_format_attributed(nx_graphs):
    """
    Convert NetworkX graphs with node attributes to grakel format.
    
    Uses the same format as load_dataset_from_files in main.py:
    [edges_list, node_attrs_dict]
    
    Parameters
    ----------
    nx_graphs : list
        List of NetworkX graphs with node 'label' and 'attributes' fields
    
    Returns
    -------
    G_grakel : list
        Graphs in grakel format suitable for attributed kernels
    """
    G_grakel = []
    
    # First pass: determine the attribute dimension from all graphs
    attr_dim = None
    for nx_graph in nx_graphs:
        for node in nx_graph.nodes():
            if 'attributes' in nx_graph.nodes[node]:
                attrs = nx_graph.nodes[node]['attributes']
                if isinstance(attrs, (list, tuple, np.ndarray)):
                    attr_dim = len(attrs)
                    break
                else:
                    attr_dim = 1
                    break
        if attr_dim is not None:
            break
    
    if attr_dim is None:
        attr_dim = 1  # Default to 1 if no attributes found
    
    for nx_graph in nx_graphs:
        # Get nodes and create mapping to 0-indexed integers
        nodes = sorted(list(nx_graph.nodes()))
        node_mapping = {node: i for i, node in enumerate(nodes)}
        
        # Convert edges to list of tuples with integer indices
        edges = []
        for u, v in nx_graph.edges():
            if u in node_mapping and v in node_mapping:
                u_idx = node_mapping[u]
                v_idx = node_mapping[v]
                edges.append((u_idx, v_idx))
        
        # Extract node attributes - using same format as main.py
        node_attrs = {}
        for i, node in enumerate(nodes):
            if node in nx_graph.nodes():
                node_data = nx_graph.nodes[node]
                if 'attributes' in node_data:
                    attr_val = node_data['attributes']
                    # Ensure attribute is in correct format
                    if isinstance(attr_val, (list, tuple, np.ndarray)):
                        # Use tuple of values for multi-dimensional attributes
                        # Ensure consistent dimension
                        attr_tuple = tuple(float(x) for x in attr_val)
                        if len(attr_tuple) != attr_dim:
                            # Pad or truncate to match expected dimension
                            if len(attr_tuple) < attr_dim:
                                attr_tuple = attr_tuple + (0.0,) * (attr_dim - len(attr_tuple))
                            else:
                                attr_tuple = attr_tuple[:attr_dim]
                        node_attrs[i] = attr_tuple
                    else:
                        # Single value - pad to match dimension
                        node_attrs[i] = (float(attr_val),) + (0.0,) * (attr_dim - 1)
                else:
                    # Default attribute if none exists
                    node_attrs[i] = (0.0,) * attr_dim
            else:
                node_attrs[i] = (0.0,) * attr_dim
        
        # Create grakel-compatible format: [edges_list, node_attrs_dict]
        G_grakel.append([edges, node_attrs])
    
    return G_grakel


def read_graph_data(dataset_name='BZR'):
    """Read graph data from text files.
    
    Parameters
    ----------
    dataset_name : str
        Name of the dataset ('BZR' or 'COX2')
    
    Returns
    -------
    tuple : (edges, graph_indicators, graph_labels, node_labels, node_attributes)
    """
    # Determine the path prefix based on dataset
    if dataset_name == 'BZR':
        prefix = "BZR"
        path_prefix = "BZR/"
    elif dataset_name == 'COX2':
        prefix = "COX2"
        path_prefix = "COX2/"
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    
    # Read the adjacency matrix (edges)
    with open(f"{path_prefix}{prefix}_A.txt", "r") as f:
        edges = [tuple(map(int, line.split(','))) for line in f]

    # Read graph indicators
    with open(f"{path_prefix}{prefix}_graph_indicator.txt", "r") as f:
        graph_indicators = [int(line.strip()) for line in f if line.strip()]

    # Read graph labels
    with open(f"{path_prefix}{prefix}_graph_labels.txt", "r") as f:
        graph_labels = [int(line.strip()) for line in f]

    # Read node labels
    with open(f"{path_prefix}{prefix}_node_labels.txt", "r") as f:
        node_labels = [int(line.strip()) for line in f]

    # Read node attributes
    with open(f"{path_prefix}{prefix}_node_attributes.txt", "r") as f:
        node_attributes = [
            list(map(float, line.strip().split(','))) for line in f if line.strip()
        ]

    return edges, graph_indicators, graph_labels, node_labels, node_attributes


def normalize_features(features):
    """Normalize feature vectors to zero mean and unit variance."""
    features = np.array(features)
    # Normalize each feature dimension independently
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0)
    # Avoid division by zero
    std[std == 0] = 1
    normalized_features = (features - mean) / std
    return normalized_features.tolist()


def construct_networkx_graph(edges, graph_indicators, graph_labels,
                              node_labels, node_attributes):
    """Construct NetworkX graphs from raw data."""
    graphs = {}
    graphs_normalized = {}

    # Normalize all node attributes
    normalized_attributes = normalize_features(node_attributes)

    for node_id, graph_id in enumerate(graph_indicators, start=1):
        if graph_id not in graphs:
            graphs[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])
            graphs_normalized[graph_id] = nx.Graph(label=graph_labels[graph_id - 1])

    for edge in edges:
        node1, node2 = edge
        graph_id = graph_indicators[node1 - 1]
        graphs[graph_id].add_edge(node1, node2)
        graphs_normalized[graph_id].add_edge(node1, node2)

    for node_id, (graph_id, node_label, node_attr,
                  norm_attr) in enumerate(zip(graph_indicators, node_labels,
                                              node_attributes,
                                              normalized_attributes),
                                          start=1):
        # Original graphs with original features
        graphs[graph_id].nodes[node_id]['label'] = node_label
        graphs[graph_id].nodes[node_id]['attributes'] = node_attr

        # Normalized graphs with normalized features
        graphs_normalized[graph_id].nodes[node_id]['label'] = node_label
        graphs_normalized[graph_id].nodes[node_id]['attributes'] = norm_attr

    return graphs, graphs_normalized


def compute_bzr_distance_matrix(graph):
    """Compute pairwise distance matrix for a graph."""
    num_nodes = len(graph.nodes)
    distance_matrix = np.full((num_nodes, num_nodes), 10000.0)

    node_list = list(graph.nodes())
    node_index = {node: i for i, node in enumerate(node_list)}

    for node in graph.nodes:
        lengths = nx.shortest_path_length(graph, source=node, weight="weight")
        for target, length in lengths.items():
            if target in node_index:
                distance_matrix[node_index[node], node_index[target]] = length

    return distance_matrix


def read_family_labels(dataset_name='BZR'):
    """Read family labels from Excel file.
    
    Parameters
    ----------
    dataset_name : str
        Name of the dataset ('BZR' or 'COX2')
    
    Returns
    -------
    list : Family labels for each graph
    """
    df = pd.read_excel(f"{dataset_name}.xlsx")
    family_labels = df["Fam"].tolist()
    return family_labels


def prepare_family_classification_data(dataset_name='BZR', min_family_size=10):
    """Prepare data for family-based classification.
    
    Parameters
    ----------
    dataset_name : str
        Name of the dataset ('BZR' or 'COX2')
    min_family_size : int
        Minimum number of graphs required in a family to include it
    
    Returns
    -------
    graphs_list_fam : list
        List of NetworkX graphs (non-normalized)
    filtered_family_labels : list
        Family labels for graphs (only families with min_family_size+ members)
    """
    print("\n" + "=" * 60)
    print(f"PREPARING {dataset_name} FAMILY CLASSIFICATION DATA")
    print("=" * 60)
    
    # Read and construct graphs
    edges, graph_indicators, graph_labels, node_labels, node_attributes = read_graph_data(dataset_name)
    graphs, graphs_normalized = construct_networkx_graph(
        edges, graph_indicators, graph_labels, node_labels, node_attributes
    )
    
    # Convert to lists
    graphs_list = []
    for graph_id, graph in graphs.items():
        graphs_list.append(graph)
    
    # Read family labels
    family_labels = read_family_labels(dataset_name)
    
    # Filter families with min_family_size+ members
    family_counts = Counter(family_labels)
    print(f"\nOriginal family label counts: {dict(family_counts)}")
    
    valid_families = {
        family for family, count in family_counts.items() if count >= min_family_size
    }
    print(f"Families with {min_family_size}+ elements: {valid_families}")
    
    # Filter graphs and labels
    valid_indices = [
        i for i, family in enumerate(family_labels) if family in valid_families
    ]
    
    filtered_family_labels = [family_labels[i] for i in valid_indices]
    graphs_list_fam = [graphs_list[i] for i in valid_indices]
    
    # Print statistics
    remaining_counts = Counter(filtered_family_labels)
    print("\nFiltered family distribution:")
    for family, count in sorted(remaining_counts.items()):
        print(f"  Family {family}: {count} graphs")
    
    print(f"\nTotal graphs for classification: {len(graphs_list_fam)}")
    print(f"Number of family classes: {len(np.unique(filtered_family_labels))}")
    
    return graphs_list_fam, filtered_family_labels


def evaluate_fgw_multiclass(graphs_list, y, log_dir, n_folds=5, n_inner=5):
    """Evaluate FGW method for multiclass classification using nested CV.
    
    Parameters
    ----------
    graphs_list : list
        List of NetworkX graphs
    y : array
        Multiclass labels
    log_dir : str
        Directory for log files
    n_folds : int
        Number of outer folds
    n_inner : int
        Number of inner folds
        
    Returns
    -------
    str : Status message
    """
    print("\n" + "=" * 60)
    print("EVALUATING FGW MULTICLASS CLASSIFICATION")
    print("=" * 60)
    
    # Setup logger
    logger_name = 'fgw_family'
    logger = setup_logger(logger_name, f'{log_dir}/fgw_family_nested_cv.log')
    
    # Convert NetworkX graphs to Graph objects for FGW
    X_fgw = []
    for nx_graph in graphs_list:
        g = Graph()
        g.nx_graph = nx_graph
        X_fgw.append(g)
    
    X_fgw = np.array(X_fgw)
    y_array = np.array(y)
    
    # FGW parameter grid
    alpha_list = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
    C_list = list(np.logspace(-4, 4, 15))
    gamma_list = list([2**k for k in np.linspace(-10, 10, 15)])
    
    tuned_parameters = [{
        'alpha': alpha_list,
        'C': C_list,
        'gamma': gamma_list,
        'features_metric': ['sqeuclidean'],
        'method': ['shortest_path'],
        'wl': [0],
        'amijo': [True]
    }]
    
    try:
        nested_fgw_multiclass(
            X_fgw,
            y_array,
            tuned_parameters,
            'BZR_family',
            logger,
            path=None,
            n_inner=n_inner,
            n_iter=n_folds,
            verbose=1
        )
        
        print("  FGW classification completed - check log file for results")
        return "Completed - check log file for results"
        
    except Exception as e:
        print(f"  Error in FGW classification: {e}")
        return f"Error: {e}"


def evaluate_graph_kernels_multiclass(graphs_list, y, log_dir, n_folds=5, n_inner=5):
    """Evaluate graph kernel methods for multiclass classification using nested CV.
    
    Parameters
    ----------
    graphs_list : list
        List of NetworkX graphs
    y : array
        Multiclass labels
    log_dir : str
        Directory for log files
    n_folds : int
        Number of outer folds
    n_inner : int
        Number of inner folds
        
    Returns
    -------
    dict : Results for each method
    """
    print("\n" + "=" * 60)
    print("EVALUATING GRAPH KERNEL METHODS FOR MULTICLASS CLASSIFICATION")
    print("=" * 60)
    
    results = {}
    y_array = np.array(y)
    
    # Convert NetworkX graphs to Grakel format with attributes
    print("Converting NetworkX graphs to Grakel format...")
    G_grakel = convert_nx_to_grakel_format_attributed(graphs_list)
    print(f"Converted {len(G_grakel)} graphs to Grakel format")
    
    # Define parameter grids for each method
    C_list = list(np.logspace(-5, 5, 15))
    tmax_list = [1, 3, 5, 8, 10, 15, 20]
    
    methods = {
        'hopper': [{
            'C': C_list,
            'kernel_params': [[{"name": "graph_hopper", "kernel_type": 'linear'}]],
            'normalize': [True]
        }],
        'propa': [{
            'C': C_list,
            'kernel_params': [[{"name": "propagation", "with_attributes": True, "t_max": x}] 
                             for x in tmax_list],
            'normalize': [True]
        }]
    }
    
    for method_name, param_grid in methods.items():
        print(f"\n--- Evaluating {method_name.upper()} method ---")
        
        # Setup logger
        logger_name = f'{method_name}_family'
        logger = setup_logger(logger_name, f'{log_dir}/{logger_name}_nested_cv.log')
        
        try:
            nested_cv_multiclass(
                G_grakel,
                y_array,
                param_grid,
                logger,
                n_iter=n_folds,
                n_inner=n_inner,
                verbose=1
            )
            
            results[method_name] = "Completed - check log file for results"
            print(f"  {method_name.upper()} classification completed - results in log file")
            
        except Exception as e:
            print(f"  Error in {method_name.upper()} classification: {e}")
            results[method_name] = f"Error: {e}"
    
    return results


def evaluate_dataset(dataset_name, n_outer_folds=5, n_inner_folds=5, min_family_size=10):
    """Evaluate a single dataset for family classification.
    
    Parameters
    ----------
    dataset_name : str
        Name of the dataset ('BZR' or 'COX2')
    n_outer_folds : int
        Number of outer CV folds
    n_inner_folds : int
        Number of inner CV folds
    min_family_size : int
        Minimum family size to include
        
    Returns
    -------
    dict : Results for all methods
    """
    print("\n" + "=" * 60)
    print(f"{dataset_name} FAMILY CLASSIFICATION - NESTED CROSS-VALIDATION")
    print("=" * 60)
    
    # Create log directory
    log_dir = f'nested_cv_results/{dataset_name.lower()}_family_classification'
    try:
        if not os.path.exists(log_dir):
            os.makedirs(log_dir)
    except OSError as e:
        print(f"Error creating log directory: {e}")
        return {}
    
    # Prepare data
    graphs_list_fam, filtered_family_labels = prepare_family_classification_data(
        dataset_name, min_family_size=min_family_size
    )
    
    if len(graphs_list_fam) == 0:
        print(f"No graphs found for {dataset_name} with family size >= {min_family_size}")
        return {'error': 'No graphs found'}
    
    # Evaluate FGW method
    fgw_result = evaluate_fgw_multiclass(
        graphs_list_fam,
        filtered_family_labels,
        log_dir,
        n_folds=n_outer_folds,
        n_inner=n_inner_folds
    )
    
    # Evaluate graph kernel methods
    kernel_results = evaluate_graph_kernels_multiclass(
        graphs_list_fam,
        filtered_family_labels,
        log_dir,
        n_folds=n_outer_folds,
        n_inner=n_inner_folds
    )
    
    # Print summary for this dataset
    print("\n" + "-" * 40)
    print(f"{dataset_name} Family Classification Results:")
    print(f"  FGW: {fgw_result}")
    for method, status in kernel_results.items():
        print(f"  {method.upper()}: {status}")
    print(f"  Log files saved in: {log_dir}/")
    
    return {'fgw': fgw_result, **kernel_results}

In [3]:
def main():
    """Main evaluation function for BZR and COX2 family classification."""
    print("\n" + "=" * 60)
    print("MULTICLASS FAMILY CLASSIFICATION - NESTED CROSS-VALIDATION")
    print("=" * 60)
    
    # Cross-validation parameters
    n_outer_folds = 5
    n_inner_folds = 5
    min_family_size = 10
    
    # Datasets to evaluate
    datasets = ['BZR', 'COX2']
    
    all_results = {}
    
    for dataset_name in datasets:
        print(f"\n{'#' * 60}")
        print(f"# Processing {dataset_name} Dataset")
        print(f"{'#' * 60}")
        
        try:
            results = evaluate_dataset(
                dataset_name,
                n_outer_folds=n_outer_folds,
                n_inner_folds=n_inner_folds,
                min_family_size=min_family_size
            )
            all_results[dataset_name] = results
        except Exception as e:
            print(f"Error processing {dataset_name}: {e}")
            all_results[dataset_name] = {'error': str(e)}
    
    # Print final summary
    print("\n" + "=" * 60)
    print("FINAL SUMMARY")
    print("=" * 60)
    print(f"Nested cross-validation completed with {n_outer_folds} outer folds and {n_inner_folds} inner folds.")
    print(f"Minimum family size: {min_family_size}")
    
    for dataset_name, results in all_results.items():
        print(f"\n{dataset_name} Dataset:")
        if 'error' in results:
            print(f"  Error: {results['error']}")
        else:
            for method, status in results.items():
                print(f"  {method.upper()}: {status}")
    
    print(f"\nDetailed results can be found in 'nested_cv_results/' directory")


if __name__ == "__main__":
    main()


MULTICLASS FAMILY CLASSIFICATION - NESTED CROSS-VALIDATION

############################################################
# Processing BZR Dataset
############################################################

BZR FAMILY CLASSIFICATION - NESTED CROSS-VALIDATION

PREPARING BZR FAMILY CLASSIFICATION DATA

Original family label counts: {'A.1': 91, 'A.2': 3, 'A.3': 1, 'A.4': 1, 'A.5': 5, 'A.6': 1, 'A.7': 1, 'A.8': 1, 'A.9': 2, 'A.10': 1, 'A.11': 4, 'A.12': 2, 'A.13': 10, 'A.14': 6, 'A.15': 10, 'A.16': 1, 'A.17': 2, 'A.18': 6, 'A.19': 1, 'A.20': 6, 'A.21': 1, 'A.22': 4, 'A.23': 5, 'A.24': 32, 'A.25': 9, 'A.26': 10, 'A.27': 1, 'B': 1, 'C': 1, 'D': 1, 'E.1': 3, 'E.2': 10, 'F': 5, 'G.1': 3, 'G.2': 6, 'H': 3, 'I': 28, 'J': 3, 'K.1': 3, 'K.2': 2, 'L.1': 2, 'L.2': 2, 'L.3': 1, 'M.1': 45, 'M.2': 1, 'M.3': 1, 'M.4': 5, 'M.5': 15, 'N.1': 40, 'N.2': 1, 'N.3': 1, 'N.4': 3, 'N.5': 1, 'N.6': 1}
Families with 10+ elements: {'A.26', 'M.5', 'A.1', 'A.13', 'A.24', 'M.1', 'E.2', 'N.1', 'A.15', 'I'}

Filtered 

/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

  FGW classification completed - check log file for results

EVALUATING GRAPH KERNEL METHODS FOR MULTICLASS CLASSIFICATION
Converting NetworkX graphs to Grakel format...
Converted 291 graphs to Grakel format

--- Evaluating HOPPER method ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graph_hopper', 'kernel_type': 'linear'}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'graph_hopper', 'k

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. U

  HOPPER classification completed - results in log file

--- Evaluating PROPA method ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'propagation', 'with_attributes': True, 't_max': 1}], [{'name': 'propagation', 'with_attributes': True, 't_max': 3}], [{'name': 'propagation', 'with_attributes': True, 't_max': 5}], [{'name': 'propagation', 'with_attributes': True, 't_max': 8}], [{'name': 'propagation', 'with_attributes': True, 't_max': 10}], [{'name': 'propagation', 'with_attributes': True, 't_max': 15}], [{'name': 'propagation', 'with_attributes': True, 't_max': 20}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 

...

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creatin

  HOPPER classification completed - results in log file

--- Evaluating PROPA method ---
explode_tuned_parameters input: [{'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 0.0071968567300115215, 0.037275937203149416, 0.19306977288832497, 1.0, 5.1794746792312125, 26.826957952797272, 138.9495494373139, 719.6856730011528, 3727.593720314938, 19306.977288832535, 100000.0], 'kernel_params': [[{'name': 'propagation', 'with_attributes': True, 't_max': 1}], [{'name': 'propagation', 'with_attributes': True, 't_max': 3}], [{'name': 'propagation', 'with_attributes': True, 't_max': 5}], [{'name': 'propagation', 'with_attributes': True, 't_max': 8}], [{'name': 'propagation', 'with_attributes': True, 't_max': 10}], [{'name': 'propagation', 'with_attributes': True, 't_max': 15}], [{'name': 'propagation', 'with_attributes': True, 't_max': 20}]], 'normalize': [True]}]
tuned_parameters2: {'C': [1e-05, 5.1794746792312125e-05, 0.0002682695795279727, 0.0013894954943731374, 

/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/Users/kait/Documents/GitHub/gwgraphs/Vayer Methods/utils.py:148: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.array(x_train),np.array(y_train),np.array(idx_train),np.array(x_test),np.array(y_test),np.array(idx_test)
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:156

  PROPA classification completed - results in log file

----------------------------------------
COX2 Family Classification Results:
  FGW: Completed - check log file for results
  HOPPER: Completed - check log file for results
  PROPA: Completed - check log file for results
  Log files saved in: nested_cv_results/cox2_family_classification/

FINAL SUMMARY
Nested cross-validation completed with 5 outer folds and 5 inner folds.
Minimum family size: 10

BZR Dataset:
  FGW: Completed - check log file for results
  HOPPER: Completed - check log file for results
  PROPA: Completed - check log file for results

COX2 Dataset:
  FGW: Completed - check log file for results
  HOPPER: Completed - check log file for results
  PROPA: Completed - check log file for results

Detailed results can be found in 'nested_cv_results/' directory


/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/graphkernels/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i